# Text Embeddings

With chunks in hand, the next problem is **finding the ones relevant to a question** — a search problem. The standard approach is **semantic search**: instead of matching keywords, compare the *meaning* of the question and each chunk using **embeddings**.

**A text embedding** is a numerical representation of meaning — you feed text into an embedding model and get back a long list of numbers (each roughly in `-1 … +1`). Each number scores some learned feature of the text. The important caveat: **we don't know what each dimension means** — it's tempting to imagine "how much this text is about oceans," but the actual features are learned during training and aren't human-interpretable. What matters is that *similar meaning → similar vectors*, which the next lesson exploits to rank chunks.

> **Setup:** Anthropic doesn't provide embeddings, so the course uses **VoyageAI**.
> 1. Sign up at voyageai.com (free to start) and get a key — see `VoyageAI API Key Directions.pdf`.
> 2. Add it to the repo-root `.env`: `VOYAGE_API_KEY="pa-..."`
> 3. `voyageai` is already in `requirements.txt` (the `%pip install` cell below is a no-op if it's installed).

In [1]:
# Already in requirements.txt; harmless if already installed
%pip install voyageai --quiet

Note: you may need to restart the kernel to use updated packages.


## Client setup

`voyageai.Client()` reads `VOYAGE_API_KEY` from the environment, so we just load the `.env` first.

In [2]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import voyageai

client = voyageai.Client()
print("VoyageAI client ready")

VoyageAI client ready


## Chunk the document (structure-based, from last lesson)

In [3]:
import os
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)


SECTION = os.path.join(os.path.dirname(find_dotenv()), "building-with-the-claude-api", "05-rag")
REPORT = os.path.join(SECTION, "report.md")

with open(REPORT, "r", encoding="utf-8") as f:
    text = f.read()

chunks = chunk_by_section(text)
print(f"{len(chunks)} section chunks")

15 section chunks


## Generate an embedding

`input_type` matters: use `"query"` for the user's question and `"document"` for stored chunks — Voyage optimizes each side of the comparison slightly differently. (The course's helper defaults to `"query"`; we'll be deliberate about it in the full-flow lesson.)

In [4]:
def generate_embedding(text, model="voyage-3-large", input_type="query"):
    result = client.embed([text], model=model, input_type=input_type)
    return result.embeddings[0]

## Embed a chunk

The result is a long list of floats — the vector for that section's meaning. We print the dimensionality and a few values rather than the whole thing.

In [5]:
embedding = generate_embedding(chunks[0])

print("dimensions:", len(embedding))
print("first 8 values:", embedding[:8])

dimensions: 1024
first 8 values: [-0.05453480780124664, 0.01431055087596178, -0.016921261325478554, 0.0005922440905123949, 0.02136913500726223, 0.037903621792793274, -0.047186143696308136, 0.0024173229467123747]


## What's next

An embedding on its own isn't useful — the power comes from **comparing** them. Two chunks with similar meaning have vectors that point in similar directions, which you measure with **cosine similarity**. That's the heart of semantic search and the core of the next lesson, **The full RAG flow**: embed every chunk once, embed the question at query time, rank chunks by similarity, and feed the top matches to Claude.

> **Codebase aside:** embeddings let you retrieve code by *intent* ("where do we validate auth tokens?") even when the code doesn't contain those words. But they blur exact identifiers — `getUserById` vs `getUserByEmail` embed almost identically — which is exactly why the later **BM25** lesson pairs lexical search with embeddings for code. Also mind cost: embedding a whole repo is many API calls, so you embed once and persist the vectors.